In [1]:
!pip3 install requests pandas streamlit

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\91709\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
import mysql.connector
connection = mysql.connector.connect(
	host = "localhost",
	user = "root",
	password = "Bindu@123",
    database = "air_tracker"
)
cursor = connection.cursor()
cursor

In [6]:
#Filling the null values in airport_delays
query = """UPDATE airport_delays SET avg_delay_min = ROUND(median_delay_min * delay_index, 0) WHERE avg_delay_min IS NULL"""
cursor.execute(query)
connection.commit()

In [20]:
# ===========================
# Query 1
# Show the total number of flights for each aircraft model
# ===========================
query1 = """
SELECT a.model, COUNT(f.flight_id) AS flight_count
FROM flights f
JOIN aircraft a ON f.aircraft_registration = a.registration
GROUP BY a.model;
"""

# ===========================
# Query 2
# List all aircraft (registration, model) with more than 5 flights
# ===========================
query2 = """
SELECT a.registration, a.model, COUNT(f.flight_id) AS flight_count
FROM flights f
JOIN aircraft a ON f.aircraft_registration = a.registration
GROUP BY a.registration, a.model
HAVING COUNT(f.flight_id) > 5;
"""

# ===========================
# Query 3
# For each airport, show name and outbound flights (>5)
# ===========================
query3 = """
SELECT ap.name, COUNT(f.flight_id) AS outbound_flights
FROM flights f
JOIN airport ap ON f.origin_iata = ap.iata_code
GROUP BY ap.name
HAVING COUNT(f.flight_id) > 5;
"""

# ===========================
# Query 4
# Find top 3 destination airports (name, city) by arrivals
# ===========================
query4 = """
SELECT ap.name, ap.city, COUNT(f.flight_id) AS arrivals
FROM flights f
JOIN airport ap ON f.destination_iata = ap.iata_code
GROUP BY ap.name, ap.city
ORDER BY arrivals DESC
LIMIT 3;
"""

# ===========================
# Query 5
# Show flight number, origin, destination, Domestic/International
# ===========================
query5 = """
SELECT f.flight_number,
       f.origin_iata,
       f.destination_iata,
       CASE 
           WHEN ap1.country = ap2.country THEN 'Domestic'
           ELSE 'International'
       END AS route_type
FROM flights f
JOIN airport ap1 ON f.origin_iata = ap1.iata_code
JOIN airport ap2 ON f.destination_iata = ap2.iata_code;
"""

# ===========================
# Query 6
# Show 5 most recent arrivals at DEL
# ===========================
query6 = """
SELECT f.flight_number,
       f.aircraft_registration,
       ap.name AS departure_airport,
       f.actual_arrival
FROM flights f
JOIN airport ap ON f.origin_iata = ap.iata_code
WHERE f.destination_iata = 'DEL'
ORDER BY f.actual_arrival DESC
LIMIT 5;
"""

# ===========================
# Query 7
# Find airports with no arriving flights
# ===========================
query7 = """
SELECT ap.name, ap.iata_code
FROM airport ap
WHERE ap.iata_code NOT IN (
    SELECT DISTINCT destination_iata FROM flights
);
"""

# ===========================
# Query 8
# For each airline, count flights by status
# ===========================
query8 = """
SELECT airline_code,
       SUM(CASE WHEN status = 'On Time' THEN 1 ELSE 0 END) AS on_time_count,
       SUM(CASE WHEN status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_count,
       SUM(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled_count
FROM flights
GROUP BY airline_code;
"""

# ===========================
# Query 9
# Show cancelled flights with aircraft and airports
# ===========================
query9 = """
SELECT f.flight_number,
       f.aircraft_registration,
       ap1.name AS origin_airport,
       ap2.name AS destination_airport,
       f.scheduled_departure
FROM flights f
JOIN airport ap1 ON f.origin_iata = ap1.iata_code
JOIN airport ap2 ON f.destination_iata = ap2.iata_code
WHERE f.status = 'Cancelled'
ORDER BY f.scheduled_departure DESC;
"""

# ===========================
# Query 10
# List city pairs with >2 different aircraft models
# ===========================
query10 = """
SELECT ap1.city AS origin_city,
       ap2.city AS destination_city,
       COUNT(DISTINCT a.model) AS model_count
FROM flights f
JOIN airport ap1 ON f.origin_iata = ap1.iata_code
JOIN airport ap2 ON f.destination_iata = ap2.iata_code
JOIN aircraft a ON f.aircraft_registration = a.registration
GROUP BY ap1.city, ap2.city
HAVING COUNT(DISTINCT a.model) > 2;
"""

# ===========================
# Query 11
# Compute % of delayed flights per destination airport
# ===========================
query11 = """
SELECT ap.name AS destination_airport,
       (SUM(CASE WHEN f.status = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS delayed_percentage
FROM flights f
JOIN airport ap ON f.destination_iata = ap.iata_code
GROUP BY ap.name
ORDER BY delayed_percentage DESC
"""



In [25]:
# Query 1 → Show the total number of flights for each aircraft model
# Query 2 → List all aircraft (registration, model) with more than 5 flights
# Query 3 → For each airport, show name and outbound flights (>5)
# Query 4 → Find top 3 destination airports (name, city) by arrivals
# Query 5 → Show flight number, origin, destination, Domestic/International
# Query 6 → Show 5 most recent arrivals at DEL
# Query 7 → Find airports with no arriving flights
# Query 8 → For each airline, count flights by status
# Query 9 → Show cancelled flights with aircraft and airports
# Query 10 → List city pairs with >2 different aircraft models
# Query 11 → Compute % of delayed flights per destination airport

cursor.execute(query10)
for row in cursor:
    print(row)

('Amsterdam', 'Bangalore', 3)
('Bangalore', 'Paris', 3)
('London', 'Bangalore', 3)
('London', 'Chennai', 3)
('Los Angeles', 'Mumbai', 3)
('New Delhi', 'Bangalore', 4)
('Paris', 'Bangalore', 4)
('Tokyo', 'New Delhi', 3)
